# DWthon: RAG pod lupą — dzień 2

## Structured Output w akcji: klasyfikujemy dane

Witam na drugim dniu. Mam nadzieję, że wczoraj udało się:

- sklonować prywatne repozytorium z GitHuba,
- postawić Obsidian i podpiąć vault,
- nawiązać pierwsze połączenie z Bielikiem.

Jeśli tak — świetnie, fundament jest gotowy. Jeśli masz wyzwanie na którymś kroku, napisz na Slacku (`#dwthon_llm_day1`). Nadrobimy.

Dzisiaj **mniej "literek", więcej kodu**. Ale nie bez myślenia — chcę, aby było zrozumiałe **dlaczego** robimy coś w określony sposób. Narzędzia się zmieniają, sposób myślenia zostaje.

### Czego nauczysz się dzisiaj (Dzień 2)

1. Zrozumiesz (lub przypomnisz sobie) **Structured Output** — czyli Poziom 2 na mapie 7 Poziomów Autonomii AI
2. Zobaczysz **dlaczego Poziom 2 jest fundamentem Poziomu 3** (RAG) — bez niego cała reszta to zamki na piasku
3. Zrozumiesz, dlaczego **nazwa pola w klasie Pydantic to część promptu**, a nie tylko etykieta dla kompilatora
4. Zbudujesz **klasyfikator dokumentów finansowych** (`FinancialDocument`) — najpierw prostą wersję, potem rozbudowaną
5. Zbudujesz **klasyfikator tabel finansowych** (`TableDetails`) — który jutro stanie się fundamentem Mapy Wiedzy w Obsidianie
6. Porównasz zachowanie **Bielika** (11B, polski — damy mu szansę) i **gpt-4o-mini** (OpenAI) na tym samym zadaniu klasyfikacji
7. Przygotujesz strukturę metadanych, z której jutro powstanie graf widoczny wyżej

### Dokąd zmierzamy (i po co dzisiaj Structured Output?)

<img src="../images/graf_example.jpg" width="900px" />

*Tak będzie wyglądała Twoja Mapa Wiedzy w Obsidianie na końcu dnia 3 — graf powiązań między spółkami, dokumentami i tabelami. Każdy węzeł to artykuł wygenerowany z metadanych, które zaczynamy tworzyć już dziś.*


### Przykładowy wierzchołek (PZU)
<img src="../images/graf_pzu.jpg" width="900px" />

Dzisiaj kładziemy fundament — **ustrukturyzowane metadane**. Jutro z tych metadanych wygenerujemy strony Markdown, wrzucimy je do Obsidiana i zobaczymy graf na żywo.

> ☝️ **Ważna uwaga praktyczna:** W ramach dzisiejszego dnia pracujemy na kilku przykładowych tabelach i jednym dokumencie. Nie uruchamiamy klasyfikacji na pełnym zbiorze (1400+ tabel) — ja to już zrobiłem za Ciebie, a gotowe dane dostaniesz jutro. Dzisiaj chodzi o **zrozumienie konceptu**, a nie o przetworzenie wszystkich danych.



## Szybki powrót do kontekstu

Pamiętasz mapę **7 Poziomów Autonomii AI**? Jeśli nie — szybkie przypomnienie:

- **Poziom 1:** Chat (99% rynku, „napisz mi maila")
- **Poziom 2:** Programowalne AI (**Structured Output** — model zwraca strukturę, nie ścianę tekstu)
- **Poziom 3:** Zintegrowane AI (**RAG** — model sięga do Twojej bazy wiedzy)
- **Poziom 4:** Kontekstowe AI (pamięć + narzędzia)
- **Poziom 5:** Strategiczne AI (planowanie)
- **Poziom 6-7:** Meta-autonomia i AGI (na razie teoria)

Wczoraj powiedzieliśmy sobie coś ważnego:

> **Nie zbudujesz dobrego Poziomu 3, jeśli nie opanujesz Poziomu 2.**

I to jet coś więcej niż tylko opinia. Przekonasz się już za moment. RAG to **system**, a nie mechanizm "wrzuć PDF do bazy wektorowej". Serce RAG-a to **reprezentacja danych**. A żeby mieć dobrą reprezentację, potrzebujesz modelu, który zwróci Ci **strukturę**, a nie opowiadanie.

### I właśnie to dzisiaj zrobimy

Weźmiemy surowe tabele finansowe i **zamienimy je w ustrukturyzowane artefakty** — z tytułem, opisem, tagami, listą pytań, na które tabela odpowiada. To jest dokładnie ten sam sposób myślenia: **write-time processing**, a nie query-time chaos.

Ale najpierw — fundament.

## Wczytujemy biblioteki

Tak jak wczoraj: jeśli nie programowałeś/aś wcześniej — **nie musisz tego kodu rozumieć w szczegółach**. Klikaj, uruchamiaj, obserwuj wynik. Eksperymentuj. Kod nie gryzie.

In [ ]:
from openai import OpenAI
from typing import List, Optional, Literal, Set
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from datetime import date
import instructor
import pandas as pd
from tqdm import tqdm

In [ ]:
load_dotenv()

In [ ]:
client = OpenAI(
    base_url="https://llm-api.dataworkshop.eu"
)

client = instructor.patch(client, mode=instructor.Mode.MD_JSON)

### Dwa modele LLM, jedna klatka

Dzisiaj będziemy pracować z **dwoma modelami**:

- **`speakleash/Bielik-11B-v3.0-Instruct`** — polski model, ten sam co wczoraj (dajmy szansę)
- **`gpt-4o-mini`** — mały, ale dość stabilny model od OpenAI (sprawdzony w boju)

Dlaczego dwa? Bo chcę, dać możliwość sprawdzać. Każdy model ma swoje mocne i słabe strony. Bielik jest aktywnie rozwijamy - więc dajemu mu szansę, ale mniejszy — przy bardziej skomplikowanych strukturach JSON... czasem się gubi. `gpt-4o-mini` jest bardzo stabilny przy `Structured Output`, więc będzie naszym głównym koniem roboczym do klasyfikacji.

W praktyce produkcyjnej to jest decyzja inżynierska: **dopasuj model do zadania, nie na odwrót**. Czasem Bielik wystarczy (proste struktury, prosty polski tekst). Czasem trzeba coś mocniejszego. Też Bielik może działać lokalnie, więc fajnie sprawdzić co możemy zrobić już teraz.

Możesz przełączać się między modelami i porównywać wyniki. To ćwiczenie niesamowicie rozwija intuicję.

## Structured Output — szybkie przypomnienie

**Structured Output** to mechanizm, który zmusza LLM do zwrócenia **dokładnie tej struktury danych**, którą definiujesz — zamiast luźnego tekstu.

Zamiast:

> *"Cześć! Oczywiście, ta książka to 'Solaris' Stanisława Lema z 1961 roku, należąca do gatunku science fiction. Czy chcesz wiedzieć coś więcej?"*

dostajesz:

```python
Book(title="Solaris", author="Stanisław Lem", year=1961, genre="science fiction")
```

Różnica? **Drugi wariant da się bezpośrednio wpiąć w kod.** Bez parsowania, bez wyrażeń regularnych, bez zgadywania.

### Zobaczmy to w akcji

In [ ]:
class Book(BaseModel):
    title: str = Field(..., description="Tytuł książki")
    author: str = Field(..., description="Autor książki")
    year: Optional[int] = Field(None, description="Rok pierwszego wydania")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=Book,
    messages=[
        {"role": "user", "content": "Solaris to powieść Stanisława Lema wydana w 1961 roku."}
    ],
)

print(f"Tytuł:  {response.title}")
print(f"Autor:  {response.author}")
print(f"Rok:    {response.year}")

### Co się właśnie stało?

1. **Definiujesz strukturę (klasę) `Book`** z trzema polami. Każde ma typ (`str`, `int`) i opis (`description`).
2. **Przekazujesz ją jako `response_model`** do `chat.completions.create()`.
3. `instructor` **przetłumaczył Twoją klasę na JSON Schema** i wysłał modelowi "formularz do wypełnienia".
4. Model **wypełnił formularz**, a `instructor` **zwrócił Ci gotowy obiekt Pythona**.

Cała magia polega na tym, że:

> 💡 **Nazwy pól (`title`, `author`, `year`) i ich opisy (`description=...`) są częścią promptu.**

Jeśli nazwiesz pole `x` i nie dodasz opisu, model będzie zgadywał. Jeśli nazwiesz je `author_full_name_as_on_book_cover` i dodasz sensowny `description`, model **zrozumie kontekst** i odpowie precyzyjniej.

To jest Software 3.0: **kod jest promptem, prompt jest kodem**.

## 🧠 Zadanie 2.0 (rozgrzewka)

Zanim ruszymy z poważną pracą — rozgrzejmy palce.

### Co masz zrobić (konkretnie):

<p>1. Skopiuj klasę <code>Book</code> powyżej do nowej komórki.</p>
<p>2. Dodaj do niej <b>przynajmniej jedno nowe pole</b>. Wybierz jedno z poniższych (lub wymyśl własne):</p>
<p>
<ul>
    <li><code>genre</code> — gatunek literacki (typ <code>Optional[str]</code>)</li>
    <li><code>main_themes</code> — lista głównych tematów (typ <code>List[str]</code>)</li>
    <li><code>country_of_origin</code> — kraj pochodzenia autora (typ <code>Optional[str]</code>)</li>
</ul>
</p>
<p>3. Zmień tekst wejściowy na dłuższy fragment (np. opis książki z Wikipedii — skopiuj 2-3 zdania).</p>
<p>4. Uruchom i wypisz wynik. Zobacz, czy model poprawnie wypełnił nowe pole.</p>
<p>5. <b>Bonus:</b> przełącz model na Bielika (<code>speakleash/Bielik-11B-v3.0-Instruct</code>) i porównaj wynik.</p>

### Dlaczego to ważne?

To jest pierwszy moment, w którym widzisz na własne oczy, jak **nazwa pola i jego opis sterują modelem**. Nazwij pole `x` i zobacz chaos. Nazwij je `main_themes` z sensownym `description` — dostaniesz sensowną listę.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

Skopiuj strukturę klasy <code>Book</code> i dodaj nowe pole. Pamiętaj o typie i opisie:

```python
genre: Optional[str] = Field(
    None, 
    description="Gatunek literacki (np. science fiction, fantasy, kryminał)"
)
```

Dla listy tematów:

```python
main_themes: List[str] = Field(
    default_factory=list, 
    description="Główne tematy poruszane w książce (3-5 elementów)"
)
```

Potem wywołanie jak wcześniej — <code>client.chat.completions.create(...)</code> z <code>response_model=Book</code>. Tylko ten sam tekst zmień na dłuższy.

</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć przykładowe rozwiązanie 👈 </summary>
<p>

```python
class Book(BaseModel):
    title: str = Field(..., description="Tytuł książki")
    author: str = Field(..., description="Autor książki")
    year: Optional[int] = Field(None, description="Rok pierwszego wydania")
    genre: Optional[str] = Field(None, description="Gatunek literacki")
    main_themes: List[str] = Field(
        default_factory=list, 
        description="Główne tematy poruszane w książce (3-5 elementów)"
    )

text = """
Solaris to powieść science fiction autorstwa Stanisława Lema, 
wydana w 1961 roku. Porusza tematy kontaktu z obcą cywilizacją, 
granic ludzkiego poznania oraz samotności istoty myślącej.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=Book,
    messages=[{"role": "user", "content": text}],
)

print(f"Tytuł:    {response.title}")
print(f"Autor:    {response.author}")
print(f"Rok:      {response.year}")
print(f"Gatunek:  {response.genre}")
print(f"Tematy:   {', '.join(response.main_themes)}")
```

<br />
<br />
<b>Komentarz:</b> Zauważ, że nie musieliśmy zmieniać promptu ("`system message`"). Zmieniliśmy tylko <b>strukturę</b> — a model sam dopasował się do nowej "listy pól do wypełnienia". To jest esencja Software 3.0: kod (struktura) staje się promptem.

</p>
</details>
</details>

In [ ]:
## YOUR CODE HERE

## Od zabawki do realnego problemu

Wyciąganie tytułu książki z jednego zdania to fajne ćwiczenie, ale w biznesie nikt nie płaci za klasyfikowanie „Solarisa". Zobaczmy coś realnego.

### Dane: 7 największych polskich spółek

W tym warsztacie pracujemy na **sprawozdaniach finansowych 7 największych spółek z GPW**:

1. **Orlen** (paliwa/energia)
2. **PKO Bank Polski** (bankowość)
3. **PGE** (energia)
4. **PZU** (ubezpieczenia)
5. **KGHM** (miedź/surowce)
6. **Tauron** (energia)
7. **Enea** (energia)

Każda z tych spółek publikuje rocznie **kilkusetstronicowe sprawozdania finansowe**. Dla analityka to kopalnia wiedzy. Dla inżyniera — wyzwanie techniczne.

### Pełna ścieżka (którą zrobiłem za Ciebie)

```
PDF (setki stron) 
   ↓ parsowanie (llama-parse)
Markdown (długi, płynny)
   ↓ segmentacja po strukturze
Fragmenty + tabele
   ↓ ekstrakcja tabel
tables_raw.parquet  ← TUTAJ ZACZYNAMY
```

> ☝️ **Uwaga:** Na pełnym DWthonie nie przerabiamy całego pipeline'u od PDF-a do parquet (to materiał na cały tydzień — robimy to w [Praktycznym LLM](https://dataworkshop.eu/pl/praktyczny-llm?utm_source=dwthon&utm_medium=lesson&utm_campaign=rag&utm_term=day2)). Skupiamy się na tej części, która jest najistotniejsza: **co zrobić z tabelami, żeby dało się z nich potem sensownie coś wyciągnąć**.

### Nasze pytania biznesowe

Po co to wszystko? Bo realny analityk chce móc zadać pytania typu:

1. *Jakie są główne źródła przychodów i jakie czynniki wpływają na ich zmienność?*
2. *Jak wygląda struktura kapitałowa i stosunek kapitału własnego do obcego?*
3. *Jakie są wskaźniki rentowności i ich zmiany w ostatnich latach?*
4. *Czy firma posiada istotne zobowiązania długoterminowe?*
5. *Jak firma zarządza ryzykiem walutowym?*

…i jeszcze kilka innych. Bez porządnej Mapy Wiedzy taki analityk musi otworzyć PDF-a, scroll-ować, szukać tabeli, potem sprawdzać w innej tabeli… godziny roboty. A my — za chwilę — zbudujemy system, który zadaje Mapie pytanie i dostaje konkretną odpowiedź ze wskazaniem źródła.

### Co mamy w danych?

1417 tabel wyciągniętych z 7 sprawozdań finansowych. Poniżej trzy przykładowe tabele (losowo wybrane), żebyś poczuł, z czym będziemy pracować.


### Tabela 1


```
Document ID:  0
Table ID:     10
```


|1.|Informacje ogólne dotyczące ENEA S.A.|
|---|---|
|Nazwa (firma):|ENEA Spółka Akcyjna|
|Forma prawna:|spółka akcyjna|
|Kraj siedziby:|Rzeczpospolita Polska|
|Siedziba:|Poznań|
|Adres:|ul. Pastelowa 8, 60-198 Poznań|
|KRS:|0000012483|
|Numer telefonu:|(+48 61) 884 55 44|
|Numer faksu:|(+48 61) 884 59 59|
|E-mail:|enea@enea.pl|
|Strona internetowa:|www.enea.pl|
|Numer klasyfikacji statystycznej (REGON):|630139960|
|Numer klasyfikacji podatkowej (NIP):|777-00-20-640|




### Tabela 2

```
Document ID:  2
Table ID:     236
```


| |31.12.2023| | |31.12.2022|
|---|---|---|---|---|
|Zaangażowanie kredytowe obejmuje kredyty, pożyczki, skupione wierzytelności, dyskonto weksli, zrealizowane gwarancje, należne odsetki oraz zaangażowanie pozabilansowe i kapitałowe|Udział w portfelu kredytowym, który obejmuje zaangażowania pozabilansowe oraz kapitałowe|Wskaźnik koncentracji zaangażowanie do wartości kapitału Tier 1 Banku|Udział w portfelu kredytowym, który obejmuje zaangażowania pozabilansowe oraz kapitałowe|Wskaźnik koncentracji zaangażowania do wartości kapitału Tier 1 Banku|
|11|31 130|8,09%|78,33%|11|27 769|7,84%|79,77%|
|21|18 422|4,79%|46,35%|21|17 134|4,84%|49,22%|
|3|4 331|1,13%|10,90%|3|5 798|1,64%|16,66%|
|41|3 582|0,93%|9,01%|41|3 688|1,04%|10,59%|
|5|3 352|0,87%|8,44%|5|2 855|0,81%|8,20%|
|Razem|60 817|15,81%|153,03%|Razem|57 244|16,17%|164,44%|


### Tabela 3

```
Document ID:  5
Table ID:     59
```


| |Patenty i licencje|Wartość firmy|Prawa majątkowe|Pozostałe|Razem|
|---|---|---|---|---|---|
|Wartość księgowa netto na 01/01/2023|1 267|-|2 825|671|4 763|
|Skumulowane umorzenie|(798)|-|-|(126)|(924)|
|Odpisy aktualizujące|(119)|-|(59)|(118)|(296)|
| |350|-|2 766|427|3 543|
|zwiększenia/(zmniejszenia) netto|(52)|-|1 652|(210)|1 390|
|Nakłady inwestycyjne|8|-|-|182|190|
|Amortyzacja|(128)|-|-|(167)|(295)|
|Koszty finansowania zewnętrznego|-|-|-|5|5|
|Połączenie jednostek|-|97|-|3|100|
|Odpisy aktualizujące netto|(67)|(97)|-|(67)|(231)|
|Pozostałe *|135|-|1 652|(166)|1 621|
| |298|-|4 418|217|4 933|
|Wartość księgowa netto na 31/12/2023|1 401|97|4 477|690|6 665|
|Skumulowane umorzenie|(917)|-|-|(288)|(1 205)|
|Odpisy aktualizujące|(186)|(97)|(59)|(185)|(527)|
| |298|-|4 418|217|4 933|


### Pułapka "to już wystarczy"

Ktoś mógłby powiedzieć: *"OK, mamy tabele w Markdownie. Wrzućmy to do bazy wektorowej, zembedujmy i koniec."*

**Nie.** I to jest centralny punkt całego tego warsztatu.

Zobacz kilka problemów, które są w tych danych:

1. **Brak nazwy firmy w tabeli** — nie ma sposobu, żeby po samej tabeli poznać, czy to sprawozdanie Orlenu czy Enei (inaczej niż po kontekście)
2. **Brak okresu** — czy to dane za 2023? 2022? Rok obrotowy kończy się w grudniu? Trudno powiedzieć bez kontekstu
3. **Różne mnożniki wartości** — jedna spółka raportuje w tysiącach PLN, inna w milionach. Jak porównywać?
4. **Identyczne tabele w różnych dokumentach** — "Bilans" jest u wszystkich, ale liczby są inne
5. **Brak semantyki** — sama tabela nie mówi *"to jest rachunek zysków i strat"* ani *"ta tabela odpowiada na pytanie o rentowność"*

Jak wrzucisz to do bazy wektorowej (grafowej) jako jeden duży worek tekstu, to dostaniesz słabe wyniki. Oczywiście możesz liczyć na cuda, ale wtedy lepiej czekaj na Mikołaja, niż na to, że dane same się uporządkują.  Wyszukiwanie *"rentowność Orlenu w 2023"* zwróciłoby chaos, bo baza wektorowa **nie wie**, która tabela dotyczy Orlenu, ani która mówi o rentowności.

### Rozwiązanie: Write-time processing

Zamiast cierpieć przy każdym zapytaniu, **raz a porządnie wzbogacamy dane metadanymi**.

Dla każdego **dokumentu** wyciągamy:

- nazwę spółki (entity_name)
- typ podmiotu (S.A., sp. z o.o. …)
- okres sprawozdania (period_start, period_end)
- kategorię dokumentu
- mnożnik wartości (tysiące / miliony PLN)

Dla każdej **tabeli** wyciągamy:

- tytuł
- opis
- tagi
- listę pytań, na które tabela odpowiada
- wnioski
- praktyczne zastosowania

Tak, to trochę przypomina to, o czym Andrej Karpathy pisał we wspomnianym poście o LLM Wiki. Ale... ze znacznie większym naciskiem na kontrolę i dopasowanie rozwiązań pod konkretny problem. Od Ciebie zależy, gdzie ustawisz suwak i co chcesz kontrolować, ale przynajmniej pokażę Ci, jakie są opcje do wyboru. Tak, to jest ten sam pomysł, który od 2 lat jest w module 3 Praktycznego LLM. **Cieszymy się, że rynek dochodzi do podobnego wniosku.**

Swoją drogą, dla zachowania równowagi warto dodać, że po komentarzach widzę, że część uczestników już wcześniej pracowała w podobny sposób. Nie zawsze mieli na to odpowiednie nazewnictwo ani spójną metodologię, ale to akurat ma drugorzędne znaczenie. Istotne jest to, że osoby, które poświęcają więcej czasu na krytyczne myślenie i zdroworozsądkową analizę, często dochodzą do bardzo podobnych wniosków. Oczywiście różni się poziom zagłębienia w temat — w DataWorkshop było to naszym świadomym celem, żeby wejść możliwie najgłębiej. Nie oznacza to jednak żadnego monopolu na wiedzę czy podejście. Wręcz przeciwnie — cieszy mnie, że rozwiązania, które mają sens, zaczynają się upowszechniać.


A teraz — do dzieła.

## Klasyfikator #1: FinancialDocument (wersja prosta)

Zaczniemy bardzo prosto. Dosłownie dwa pola.

In [ ]:
class SimpleFinancialDocument(BaseModel):
    entity_name: str = Field(
        ..., 
        description="Nazwa spółki, której dotyczy sprawozdanie. Długie formy skracaj gdzie to możliwe (np. 'Powszechny Zakład Ubezpieczeń' → 'PZU')."
    )
    title: str = Field(
        ..., 
        description="Tytuł dokumentu, np. 'Roczne sprawozdanie finansowe za 2023'"
    )

### Budujemy funkcję klasyfikującą

Funkcja przyjmie jakiś kontekst (w praktyce: początkowe tabele dokumentu, bo tam zwykle są metadane) i zwróci obiekt `SimpleFinancialDocument`.

In [ ]:
def classify_document(doc, model="gpt-4o-mini", response_model=SimpleFinancialDocument):
    system_prompt = f"""
Jesteś inteligentnym asystentem finansowym specjalizującym się w analizie i klasyfikacji dokumentów finansowych.
Twoim zadaniem jest poprawnie sklasyfikować dokument na podstawie dostarczonego kontekstu.

Zasady:
1. Klasyfikacja musi być oparta wyłącznie na dostarczonych danych.
2. Jeśli nie jesteś pewny ustaw wartość None.
3. Nazwy spółek normalizuj do popularnej, krótkiej formy.

Sklasyfikuj dokument finansowy na podstawie kontekstu.


Dokument:
"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": doc},
        ],
        response_model=response_model,
    )
    
    return response

### Przykładowy dokument

Tutaj jest przykładowa zawartość jednej ze spółek, którą będziemy analizować. Idea jest taka: bierzemy kawałek informacji z samego początku, bo akurat w takich dokumentach tam jest to, czego szukamy – nazwa firmy i inne przydatne dla nas informacje.

To jest przykładowy fragment, aby było z czym dalej pracować.

In [ ]:
random_doc = """
# TAURON Polska Energia S.A.

# Sprawozdanie finansowe

# zgodne z Międzynarodowymi Standardami Sprawozdawczości Finansowej

# zatwierdzonymi przez Unię Europejską

# za rok zakończony dnia 31 grudnia 2023 roku
---
# TAURON Polska Energia S.A

# Sprawozdanie finansowe za rok zakończony dnia 31 grudnia 2023 roku zgodne z MSSF zatwierdzonymi przez UE

# (w mln PLN)

# SPRAWOZDANIE Z CAŁKOWITYCH DOCHODÓW

# SPRAWOZDANIE Z SYTUACJI FINANSOWEJ

# SPRAWOZDANIE ZE ZMIAN W KAPITALE WŁASNYM

# SPRAWOZDANIE Z PRZEPŁYWÓW PIENIĘŻNYCH

# INFORMACJE O TAURON POLSKA ENERGIA S.A. ORAZ PODSTAWA SPORZĄDZENIA SPRAWOZDANIA FINANSOWEGO

# 1. Informacje ogólne o TAURON Polska Energia S.A.

# 2. Udziały i akcje w jednostkach powiązanych

# 3. Oświadczenie o zgodności

# 4. Kontynuacja działalności

# 5. Waluta funkcjonalna i waluta prezentacji

# 6. Zasady (polityka) rachunkowości oraz istotne wartości oparte na profesjonalnym osądzie i szacunkach

# 7. Opublikowane standardy i zmiany do standardów, które nie weszły w życie do dnia bilansowego

# 8. Zmiany stosowanych zasad rachunkowości i zmiany prezentacji

# 8.1. Zastosowanie nowych standardów i zmian do standardów

# 8.2. Przekształcenie danych porównywalnych

# 9. Zmiany klimatu i ich wpływ na stosowane zasady rachunkowości

# 10. Informacje dotyczące segmentów działalności

# 10.1. Segmenty operacyjne

# 10.2. Geograficzne obszary działalności

# 11. Utrata wartości aktywów finansowych

# 12. Pożyczki udzielone

# 13. Koszty według rodzaju

# 14. Koszty świadczeń pracowniczych

# 15. Przychody i koszty finansowe

# 16. Podatek dochodowy

# 17. Zysk (strata) przypadający na jedną akcję

# 18. Nieruchomości inwestycyjne

# 19. Prawa do użytkowania aktywów

# 20. NOTY OBJAŚNIAJĄCE DO SPRAWOZDANIA Z CAŁKOWITYCH DOCHODÓW

# 21. Udziały i akcje

# 22. Instrumenty pochodne i rachunkowość zabezpieczeń

# 23. Pozostałe aktywa finansowe

# 24. Pozostałe aktywa niefinansowe

# 25. Odroczony podatek dochodowy

# 26. Zapasy

# 27. Należności od odbiorców
---
# TAURON Polska Energia S.A

# Sprawozdanie finansowe za rok zakończony dnia 31 grudnia 2023 roku zgodne z MSSF zatwierdzonymi przez UE

(w mln PLN)
(w mln PLN)
(w mln PLN)
(w mln PLN)
(w mln PLN)

# 34. Pozostałe zobowiązania niefinansowe

....................................................................................................................... 44

# 35. Rezerwa na umowę rodzącą obciążenia

....................................................................................................................... 45

# 36. Pozostałe rezerwy, rozliczenia międzyokresowe i dotacje rządowe

....................................................................................................................... 45

# NOTY OBJAŚNIAJĄCE DO SPRAWOZDANIA Z PRZEPŁYWÓW PIENIĘŻNYCH

....................................................................................................................... 46

# 37. Istotne pozycje sprawozdania z przepływów pieniężnych

....................................................................................................................... 46

# 37.1. Przepływy środków pieniężnych z działalności operacyjnej

....................................................................................................................... 46

# 37.2. Przepływy środków pieniężnych z działalności inwestycyjnej

....................................................................................................................... 46

# 37.3. Przepływy środków pieniężnych z działalności finansowej

....................................................................................................................... 47

# INSTRUMENTY FINANSOWE I ZARZĄDZANIE RYZYKIEM FINANSOWYM

....................................................................................................................... 47

# 38. Instrumenty finansowe

....................................................................................................................... 47

# 38.1. Wartość bilansowa i wartość godziwa kategorii i klas instrumentów finansowych

....................................................................................................................... 48

# 38.2. Pozycje przychodów, kosztów, zysków i strat ujęte w sprawozdaniu z całkowitych dochodów w podziale na kategorie instrumentów finansowych

....................................................................................................................... 50

# 39. Cele i zasady zarządzania ryzykiem finansowym

....................................................................................................................... 51

# 39.1. Ryzyko kredytowe

....................................................................................................................... 52

# 39.1.1. Ryzyko kredytowe związane z należnościami od odbiorców

....................................................................................................................... 52

# 39.1.2. Ryzyko kredytowe związane z udzielonymi pożyczkami

....................................................................................................................... 53

# 39.1.3. Ryzyko kredytowe związane z innymi aktywami finansowymi

....................................................................................................................... 54

# 39.2. Ryzyko związane z płynnością

....................................................................................................................... 54

# 39.3. Ryzyko rynkowe

....................................................................................................................... 56

# 39.3.1. Ryzyko stopy procentowej

....................................................................................................................... 56

# 39.3.2. Ryzyko walutowe

....................................................................................................................... 58

# 39.3.3. Ryzyko zmian cen surowców i towarów w odniesieniu do towarowych instrumentów pochodnych

....................................................................................................................... 60

# 40. Ryzyko operacyjne

....................................................................................................................... 60

# 41. Zobowiązania warunkowe

....................................................................................................................... 61

# 42. Zabezpieczenia spłaty zobowiązań

....................................................................................................................... 64

# 43. Zobowiązania inwestycyjne

....................................................................................................................... 65

# 44. Informacja o podmiotach powiązanych

....................................................................................................................... 65

# 44.1. Transakcje z udziałem spółek powiązanych oraz spółek Skarbu Państwa

....................................................................................................................... 65

# 44.2. Wynagrodzenie kadry kierowniczej

....................................................................................................................... 67

# 45. Zarządzanie kapitałem i finansami

....................................................................................................................... 67

# 46. Struktura zatrudnienia

....................................................................................................................... 67

# 47. Wynagrodzenie biegłego rewidenta lub podmiotu uprawnionego

....................................................................................................................... 67

# 48. Podział sprawozdania finansowego na rodzaje działalności zgodnie z Art. 44 Prawa energetycznego

....................................................................................................................... 67

# 49. Pozostałe istotne informacje

....................................................................................................................... 70

# 50. Zdarzenia następujące po dniu bilansowym

....................................................................................................................... 72
---
# TAURON Polska Energia S.A

# Sprawozdanie finansowe za rok zakończony dnia 31 grudnia 2023 roku zgodne z MSSF zatwierdzonymi przez UE

# (w mln PLN)

# SPRAWOZDANIE Z CAŁKOWITYCH DOCHODÓW

|Rok zakończony|31 grudnia 2023|31 grudnia 2022|
|---|---|---|
|Przychody ze sprzedaży|34 314|27 545|
|Koszt sprzedanych towarów i usług|(33 924)|(27 160)|
|Zysk brutto ze sprzedaży|390|385|
|Koszty sprzedaży|(33)|(32)|
|Koszty ogólnego zarządu|(116)|(145)|
|Pozostałe przychody i koszty operacyjne|(11)|(8)|
|Zysk operacyjny|230|200|
|Przychody z tytułu dywidendy|475|1 797|
|Przychody odsetkowe od pożyczek|947|544|
|Koszty odsetkowe od zadłużenia|(928)|(654)|
|Aktualizacja wartości udziałów i akcji|(271)|(48)|
|Pozostałe przychody i koszty finansowe|(330)|(341)|
|Zysk (strata) przed opodatkowaniem|(637)|390|
|Podatek dochodowy|(33)|(1)|
|Zysk (strata) netto|(638)|(870)|

# Zysk (strata) na jedną akcję (w złotych):

|Typ|Wartość|
|---|---|
|Podstawowy z zysku (straty) netto|(0,36)|
|Rozwodniony z zysku (straty) netto|(0,36)|
"""

### Uruchamiamy klasyfikator

In [ ]:
result = classify_document(random_doc) 
print(f"Nazwa spółki:  {result.entity_name}")
print(f"Tytuł:         {result.title}")

### Zadziałało 🎉

Model z powodzeniem wyciągnął z tabel nazwę spółki i tytuł dokumentu.

Teraz ważne pytanie: **czy Bielik też da radę?** Sprawdźmy.

# Porównanie z Bielikiem

In [ ]:
result_bielik = classify_document(
    random_doc, 
    model="speakleash/Bielik-11B-v3.0-Instruct"
)
print(f"[Bielik]  Nazwa spółki:  {result_bielik.entity_name}")
print(f"[Bielik]  Tytuł:         {result_bielik.title}")

### Co zauważasz?

Oba modele powinny dać podobny wynik dla prostej struktury. Gdy dołożymy więcej pól, różnice zaczną być widoczne. To normalne - Bielik ma 11 miliardów parametrów, gpt-4o-mini prawdopodobnie znacznie więcej (OpenAI nie ujawnia dokładnej liczby). Ale Bielik może być 100% lokalny. To jest decyzja inżynierska na każdym projekcie.

## 🧠 Zadanie 2.1: Rozszerz klasyfikator dokumentu

### Co masz zrobić (konkretnie):

<p>1. Stwórz <b>nową klasę</b> <code>FinancialDocumentV2</code>, która zawiera:</p>
<p>
<ul>
    <li><code>entity_name</code> — nazwa spółki (tak jak w <code>SimpleFinancialDocument</code>)</li>
    <li><code>title</code> — tytuł dokumentu (tak jak w <code>SimpleFinancialDocument</code>)</li>
    <li><code>period_end</code> — <b>NOWE:</b> data zakończenia okresu sprawozdawczego (typ <code>date</code>)</li>
    <li><code>entity_type</code> — <b>NOWE:</b> forma prawna spółki, ograniczona do wartości <code>'S.A.'</code>, <code>'sp. z o.o.'</code> lub <code>'inne'</code> (typ <code>Literal</code>)</li>
</ul>
</p>
<p>2. Uruchom <code>classify_document(random_doc, response_model=FinancialDocumentV2)</code>.</p>
<p>3. Wypisz wszystkie cztery pola wyniku.</p>

### Dlaczego `Literal` to gamechanger

`Literal` zamiast zwykłego `str` to prosty, ale potężny trik — **zmusza model do wybrania jednej z podanych wartości**. Bez tego dostałbyś wariacje: "Spółka Akcyjna", "spółka akcyjna", "S.A", "S.A." — i musiałbyś to potem ręcznie sprzątać. `Literal` eliminuje ten problem **na poziomie kontraktu**, nie na poziomie promptu. Struktura steruje modelem.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

<b>Krok 1:</b> Import <code>date</code> już mamy na górze notebooka (<code>from datetime import date</code>), więc nie musisz importować ponownie.

<b>Krok 2:</b> Skopiuj kod klasy <code>SimpleFinancialDocument</code> i zmień nazwę na <code>FinancialDocumentV2</code>.

<b>Krok 3:</b> Dla pola <code>Literal</code> wzorzec wygląda tak:

```python
entity_type: Literal['S.A.', 'sp. z o.o.', 'inne'] = Field(
    ..., 
    description="Forma prawna spółki. Wybierz jedną z listy."
)
```

<b>Krok 4:</b> Dla pola typu <code>date</code>:

```python
period_end: date = Field(
    ..., 
    description="Data zakończenia okresu sprawozdawczego (np. 2023-12-31)"
)
```

<b>Krok 5:</b> Wywołanie klasyfikatora:

```python
result = classify_document(random_doc, response_model=FinancialDocumentV2)
print(result)
```

</p>

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć przykładowe rozwiązanie 👈 </summary>
<p>

```python
class FinancialDocumentV2(BaseModel):
    entity_name: str = Field(
        ..., 
        description="Nazwa spółki (skrócona forma, np. 'PZU')"
    )
    title: str = Field(
        ..., 
        description="Tytuł dokumentu, np. 'Roczne sprawozdanie finansowe za 2023'"
    )
    period_end: date = Field(
        ..., 
        description="Data zakończenia okresu sprawozdawczego (np. 2023-12-31)"
    )
    entity_type: Literal['S.A.', 'sp. z o.o.', 'inne'] = Field(
        ..., 
        description="Forma prawna spółki. Dla spółek akcyjnych wybierz 'S.A.'"
    )


result = classify_document(
    random_doc, 
    model="gpt-4o-mini",
    response_model=FinancialDocumentV2
)

print(f"Nazwa spółki:  {result.entity_name}")
print(f"Tytuł:         {result.title}")
print(f"Okres do:      {result.period_end}")
print(f"Forma prawna:  {result.entity_type}")
```
<br />
<br />
<b>Komentarz:</b>

Model powinien poprawnie wyciągnąć datę <code>2023-12-31</code> oraz formę prawną <code>S.A.</code>. Zauważ, że w prompcie systemowym nic nie zmienialiśmy — wystarczyło zmienić strukturę. <b>Struktura jest promptem.</b>

Warto teraz sprawdzić dwie rzeczy:

<p>1. <b>Co robi Bielik?</b> Przełącz model na <code>speakleash/Bielik-11B-v3.0-Instruct</code> i porównaj wyniki. Mniejszy model może się łatwiej pogubić przy <code>Literal</code> albo formacie daty. To jest właśnie moment, w którym widać, czemu wybór modelu to decyzja inżynierska.</p>

<p>2. <b>Inny tekst.</b> Spróbuj na fragmencie z innej spółki (możesz skopiować kawałek z dowolnej tabeli wyżej, zamieniając go w mini-dokument). Zobacz, czy klasyfikator działa spójnie.</p>

</p>
</details>
</details>

In [ ]:
## YOUR CODE HERE

## ⭐ Zadanie z gwiazdką: pełny FinancialDocument (tylko dla chętnych)

Jeśli masz więcej doświadczenia w programowaniu lub po prostu czujesz, że prosta wersja to za mało — spójrz na **pełną wersję** klasyfikatora, której używamy w [Praktycznym LLM](https://dataworkshop.eu/pl/praktyczny-llm?utm_source=dwthon). To nie jest obowiązkowe do zaliczenia dnia, ale daje pełen obraz, co jest możliwe, gdy zaczniesz wyciskać Structured Output do granic.

### Zwróć uwagę na 3 rzeczy w pełnej wersji:

1. **`Literal` w `category`** — model musi wybrać z listy 14 predefiniowanych kategorii. To eliminuje wariacje typu "sprawozdanie roczne" vs "Raport Roczny" vs "roczny raport".
2. **`value_multiplier`** — fundamentalne dla finansów. Bez tego nie da się porównać liczb między spółkami (jedna raportuje w tysiącach, druga w milionach).
3. **`confidence_score`** — audytowalność. Model sam ocenia, na ile jest pewien swojej odpowiedzi. Jeśli < 0.7 — warto się przyjrzeć ręcznie.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć pełny kod klasyfikatora 👈 </summary>
<p>

```python
class FinancialDocument(BaseModel):
    title: str = Field(..., description="Tytuł dokumentu, np. 'Roczne sprawozdanie finansowe za 2023'")

    entity_name: str = Field(
        ..., 
        description=(
            "Znormalizowana nazwa jednostki, której dotyczy sprawozdanie. "
            "Długie formy nazw skracamy tam, gdzie to możliwe, np. 'Powszechny Zakład Ubezpieczeń' do 'PZU'."
        )
    )
    
    entity_type: Literal['S.A.', 'sp. z o.o.', 'spółka jawna', 'spółka komandytowa', 'spółka partnerska'] = Field(
        ..., 
        description=(
            "Rodzaj podmiotu, np. 'S.A.' (Spółka Akcyjna), 'sp. z o.o.' (Spółka z ograniczoną odpowiedzialnością). "
            "Zaleca się stosowanie jednolitej formy zapisu zgodnie z podanymi wartościami."
        )
    )

    period_start: date = Field(..., description="Początek okresu finansowego, którego dotyczy sprawozdanie")
    period_end: date = Field(..., description="Koniec okresu finansowego, którego dotyczy sprawozdanie")

    category: Optional[Literal[
        'List Prezesa Zarządu', 
        'Sprawozdanie niezależnego biegłego rewidenta', 
        'Jednostkowe Sprawozdanie finansowe za rok obrotowy', 
        'Sprawozdanie Zarządu z działalności', 
        'Sprawozdanie na temat informacji niefinansowych', 
        'Oświadczenie Rady Nadzorczej', 
        'Skonsolidowane Sprawozdanie Finansowe za rok obrotowy', 
        'Bilans', 
        'Rachunek zysków i strat', 
        'Informacja dodatkowa', 
        'Raport Rady Nadzorczej', 
        'Raport Roczny', 
        'Noty Objaśniające'
    ]] = Field(
        None, 
        description="Kategoria dokumentu. Wybierz najlepiej pasującą wartość z listy."
    )

    value_multiplier: Optional[int] = Field(
        None, 
        description=(
            "Współczynnik wartości liczbowych w dokumencie. "
            "1 = złotówki, 1000 = tysiące, 1000000 = miliony. "
            "Wartość ustalana na podstawie treści dokumentu, np. '(w tys. zł)' → 1000."
        )
    )
    
    confidence_score: Optional[float] = Field(
        None, ge=0.0, le=1.0, 
        description="Pewność klasyfikacji od 0.0 do 1.0"
    )


# Wywołanie:
result = classify_document(
    random_doc, 
    model="gpt-4o-mini",
    response_model=FinancialDocument
)

print(f"Tytuł:           {result.title}")
print(f"Spółka:          {result.entity_name} ({result.entity_type})")
print(f"Okres:           {result.period_start} → {result.period_end}")
print(f"Kategoria:       {result.category}")
print(f"Mnożnik:         {result.value_multiplier}")
print(f"Pewność:         {result.confidence_score}")
```

<br />
<br />
<b>To jest ten sam kod, który na produkcji klasyfikuje setki dokumentów finansowych.</b> Nic magicznego — po prostu dobrze przemyślana struktura + prompt + `gpt-4o-mini`.

</p>
</details>

## Klasyfikator #2: TableDetails (wersja prosta)

Teraz zejdziemy o poziom niżej — do poszczególnych tabel. Dla każdej tabeli chcemy wyciągnąć:

- Co to za tabela? (tytuł)
- O czym jest? (opis)
- Jakie tagi ją opisują?
- Na jakie pytania odpowiada? (kluczowe dla retrieval!)

Zaczynamy od prostej wersji.

In [ ]:
class SimpleTableDetails(BaseModel):
    title: str = Field(..., description="Krótki tytuł tabeli (maksymalnie 10 słów)")
    description: str = Field(..., description="Opis treści i celu tabeli (2-3 zdania)")
    tags: List[str] = Field(
        default_factory=list, 
        description="Lista tagów (3-7) opisujących zawartość tabeli"
    )


In [ ]:
def describe_table(table_markdown, model="gpt-4o-mini", response_model=SimpleTableDetails):
    system_prompt = """
Jesteś asystentem finansowym specjalizującym się w analizie tabel finansowych.

Twoje zadania:
1. Zrozumieć, o czym jest dana tabela — jakie dane zawiera i w jakim celu.
2. Nadać tabeli krótki, konkretny tytuł.
3. Opisać ją zwięźle i precyzyjnie.
4. Dobrać tagi, które pomogą odnaleźć tę tabelę w bazie wiedzy.

Zasady:
- Tagi powinny być małymi literami, po polsku, np. 'przychody', 'rentowność', 'bilans'.
- Opis powinien być zrozumiały dla osoby nietechnicznej.
- Jeśli tabela jest mało informatywna (np. spis treści) — to też warto zauważyć w opisie.


Opisz tę tabelę finansową:\n\n
"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": table_markdown},
        ],
        response_model=response_model,
    )
    
    return response

### Uruchamiamy na jednej tabeli


Np. dla takiej (wylosowana):

|Dłużne aktywa finansowe wyceniane w zamortyzowanym koszcie|Stan na 31 grudnia 2023|Stan na 31 grudnia 2022|
|---|---|---|
|Krótkoterminowe dłużne aktywa finansowe wyceniane w zamortyzowanym koszcie| | |
|Obligacje wewnątrzgrupowe|214 507|231 068|
|Udzielone pożyczki|742 584|83 056|
|Krótkoterminowe dłużne aktywa finansowe wyceniane w zamortyzowanym koszcie razem|957 091|314 124|
|Długoterminowe dłużne aktywa finansowe wyceniane w zamortyzowanym koszcie| | |
|Obligacje wewnątrzgrupowe|1 077 095|1 280 101|
|Udzielone pożyczki|6 144 606|4 967 245|
|Długoterminowe dłużne aktywa finansowe wyceniane w zamortyzowanym koszcie razem|7 221 701|6 247 346|
|Razem|8 178 792|6 561 470|

In [ ]:
sample_table_markdown = """

|Dłużne aktywa finansowe wyceniane w zamortyzowanym koszcie|Stan na 31 grudnia 2023|Stan na 31 grudnia 2022|
|---|---|---|
|Krótkoterminowe dłużne aktywa finansowe wyceniane w zamortyzowanym koszcie| | |
|Obligacje wewnątrzgrupowe|214 507|231 068|
|Udzielone pożyczki|742 584|83 056|
|Krótkoterminowe dłużne aktywa finansowe wyceniane w zamortyzowanym koszcie razem|957 091|314 124|
|Długoterminowe dłużne aktywa finansowe wyceniane w zamortyzowanym koszcie| | |
|Obligacje wewnątrzgrupowe|1 077 095|1 280 101|
|Udzielone pożyczki|6 144 606|4 967 245|
|Długoterminowe dłużne aktywa finansowe wyceniane w zamortyzowanym koszcie razem|7 221 701|6 247 346|
|Razem|8 178 792|6 561 470|

"""

### Robimy opis tabeli
Używając `describe_table`

In [ ]:
result = describe_table(sample_table_markdown)

print(f"Tytuł:       {result.title}")
print(f"Opis:        {result.description}")
print(f"Tagi:        {', '.join(result.tags)}")

### Czujesz różnicę?

Przed klasyfikacją mieliśmy tylko **surowy markdown** — ściana tekstu z `|` i `---`.

Po klasyfikacji mamy:

- **Tytuł** — człowiek (i maszyna) od razu wie, o co chodzi
- **Opis** — dla osoby nietechnicznej
- **Tagi** — dla wyszukiwania

**To jest Twój pierwszy artefakt Mapy Wiedzy.** Jutro z takich artefaktów zbudujemy artykuły w Obsidianie z powiązaniami.

## 🧠 Zadanie 2.2: Rozszerz klasyfikator tabel

### Co masz zrobić (konkretnie):

<p>1. Stwórz <b>nową klasę</b> <code>TableDetailsV2</code>, która zawiera:</p>
<p>
<ul>
    <li><code>title</code>, <code>description</code>, <code>tags</code> — tak jak w <code>SimpleTableDetails</code></li>
    <li><code>questions</code> — <b>NOWE:</b> lista od 3 do 7 pytań biznesowych, na które ta tabela odpowiada (typ <code>List[str]</code>)</li>
</ul>
</p>
<p>2. Uruchom <code>describe_table(sample_table_markdown, response_model=TableDetailsV2)</code>.</p>
<p>3. Wypisz tytuł, opis, tagi oraz <b>każde pytanie w osobnej linii</b> (pamiętaj, że <code>questions</code> to lista, użyj pętli <code>for</code>).</p>

### Dlaczego pytania to złoto

Jutro, przy retrieval (Poziom 3), użytkownik zapyta: *"Jakie są główne źródła przychodów Orlenu?"*. System musi dopasować to pytanie do **właściwej tabeli** spośród 1400+ dostępnych. 

Jeśli w metadanych każdej tabeli masz listę **pytań, na które ona odpowiada**, retrieval staje się trywialny — porównujesz pytanie użytkownika z listami pytań w metadanych (semantycznie lub tekstowo) i znajdujesz najlepsze dopasowanie. 

To prosta zmiana, ale **fundamentalnie zmienia jakość systemu**. Większość naiwnych RAG-ów nie robi tego kroku — i dlatego nie działa.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

<b>Krok 1:</b> Skopiuj kod klasy <code>SimpleTableDetails</code>, zmień nazwę na <code>TableDetailsV2</code>.

<b>Krok 2:</b> Dodaj pole <code>questions</code>:

```python
questions: List[str] = Field(
    default_factory=list, 
    description="Lista od 3 do 7 pytań biznesowych, na które ta tabela odpowiada. Sformułuj pytania tak, jak zadałby je analityk finansowy."
)
```

<b>Krok 3:</b> Wywołaj funkcję:

```python
result = describe_table(sample_table_markdown, response_model=TableDetailsV2)
```

<b>Krok 4:</b> Wypisz wyniki — zwróć uwagę, że <code>result.questions</code> to lista, więc iteruj:

```python
for q in result.questions:
    print(f"  - {q}")
```

</p>

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć przykładowe rozwiązanie 👈 </summary>
<p>

```python
class TableDetailsV2(BaseModel):
    title: str = Field(..., description="Krótki tytuł tabeli (max 10 słów)")
    description: str = Field(..., description="Opis treści i celu tabeli (2-3 zdania)")
    tags: List[str] = Field(
        default_factory=list, 
        description="Lista tagów (3-7) opisujących zawartość tabeli"
    )
    questions: List[str] = Field(
        default_factory=list, 
        description="Lista od 3 do 7 pytań biznesowych, na które ta tabela odpowiada. Sformułuj pytania tak, jak zadałby je analityk finansowy."
    )


result = describe_table(
    sample_table_markdown, 
    model="gpt-4o-mini",
    response_model=TableDetailsV2
)

print(f"Tytuł:    {result.title}")
print(f"Opis:     {result.description}")
print(f"Tagi:     {', '.join(result.tags)}")
print(f"\nPytania:")
for q in result.questions:
    print(f"  - {q}")
```

<br />
<br />
<b>Komentarz:</b>

Zwróć uwagę, jak trafne są pytania, które model generuje. To nie są generyczne "jakie dane są w tabeli?". To są konkretne pytania biznesowe, typu:

<p><i>"Jak kształtowała się wartość obligacji wewnątrzgrupowych między 2022 a 2023 rokiem?"</i></p>

<p><i>"Jaki jest stosunek krótkoterminowych do długoterminowych aktywów finansowych?"</i></p>

<p>I o to chodzi. Jutro, gdy użytkownik zapyta <i>"Jak zmieniły się obligacje wewnątrzgrupowe?"</i>, system natychmiast zrozumie, do której tabeli sięgnąć — bo pytanie użytkownika <b>semantycznie pasuje</b> do jednego z pytań wygenerowanych w metadanych.</p>

<p>To jest różnica między <b>"Wrzuć wszystko do bazy wektorowej i licz na cud"</b> a <b>"Zaprojektuj reprezentację tak, żeby retrieval był prosty"</b>.</p>

</p>
</details>
</details>

In [ ]:
## YOUR CODE HERE

## ⭐ Zadanie z gwiazdką: pełny TableDetails (tylko dla chętnych)

Podobnie jak wyżej — pełna wersja klasyfikatora tabel, której używamy w [Praktycznym LLM](https://dataworkshop.eu/pl/praktyczny-llm?utm_source=dwthon). Dla osób z doświadczeniem, które chcą zobaczyć, dokąd to może pójść.

### Co dodaje pełna wersja ponad `SimpleTableDetails`:

- **`columns`** — opis każdej kolumny (nazwa, typ danych, jednostka)
- **`rows`** — liczba wierszy
- **`currency`** — waluta (`Literal['PLN', 'EUR', 'USD']`)
- **`questions`** — 5-10 pytań, na które tabela odpowiada
- **`practical_applications`** — do czego te dane można praktycznie użyć
- **`key_insights`** — kluczowe wnioski z danych

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć pełny kod klasyfikatora 👈 </summary>
<p>

```python
class ColumnDescription(BaseModel):
    name: str = Field(..., description="Nazwa kolumny") 
    data_type: str = Field(..., description='Typ danych: "string", "integer", "float", "date"')
    unit: Optional[str] = Field(None, description='Jednostka miary (np. "PLN", "EUR", "szt.")')


class TableDetails(BaseModel):
    title: str = Field(..., description="Tytuł tabeli")
    description: str = Field(..., description="Krótki opis treści i celu tabeli")
    tags: List[str] = Field(
        default_factory=list, 
        description="Lista tagów (małe litery, po polsku) opisujących zawartość tabeli"
    )
    columns: List[ColumnDescription] = Field(..., description="Opis kolumn tabeli")
    rows: int = Field(..., description="Liczba wierszy danych w tabeli")
    currency: Optional[Literal['PLN', 'EUR', 'USD']] = Field(
        None, 
        description="Waluta używana w tabeli (jeśli dotyczy)"
    )
    questions: List[str] = Field(
        default_factory=list, 
        description="Od 5 do 10 najważniejszych pytań, na które tabela odpowiada"
    )
    practical_applications: List[str] = Field(
        ..., 
        description="Praktyczne zastosowania danych (np. analiza rentowności, prognozowanie, optymalizacja kosztów)"
    )
    key_insights: Optional[List[str]] = Field(
        None, 
        description="Kluczowe wnioski wynikające z analizy tabeli, pomocne w podejmowaniu decyzji"
    )


def describe_table_full(table_markdown, model="gpt-4o-mini", response_model=TableDetails):
    system_prompt = """
Jesteś asystentem finansowym specjalizującym się w analizie danych tabelarycznych.

Twoje główne zadania:
1. Analiza danych — identyfikacja kluczowych informacji finansowych i biznesowych oraz formułowanie wniosków wspierających decyzje.
2. Interpretacja danych — wyjaśnianie znaczenia kolumn w kontekście ich typów danych, jednostek miary i roli w analizie finansowej.
3. Praktyczne zastosowania — opracowywanie rekomendacji, takich jak analiza rentowności, prognozowanie, optymalizacja kosztów czy ocena ryzyka.
4. Zadawanie pytań — sugerowanie dodatkowych analiz i wskazywanie luk lub problemów w danych.
5. Ocena jakości danych — identyfikowanie potencjalnych ograniczeń, takich jak brak spójności, niekompletność czy błędne wartości.

Ważne: Twoje odpowiedzi powinny być zrozumiałe, precyzyjne i skoncentrowane na praktycznych wnioskach, które wspierają decyzje finansowe.
"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Opisz tabele jak najlepiej:\n\n{table_markdown}"},
        ],
        response_model=response_model,
    )
    
    return response


# Wywołanie:
result_full = describe_table_full(sample_table_markdown, model="gpt-4o-mini")

print(f"Tytuł:       {result_full.title}")
print(f"Opis:        {result_full.description}")
print(f"Tagi:        {', '.join(result_full.tags)}")
print(f"Waluta:      {result_full.currency}")
print(f"Wiersze:     {result_full.rows}")
print(f"\nKolumny:")
for col in result_full.columns:
    print(f"  - {col.name} ({col.data_type}, {col.unit})")
print(f"\nPytania:")
for q in result_full.questions:
    print(f"  - {q}")
print(f"\nZastosowania praktyczne:")
for app in result_full.practical_applications:
    print(f"  - {app}")
if result_full.key_insights:
    print(f"\nKluczowe wnioski:")
    for ki in result_full.key_insights:
        print(f"  - {ki}")
```


<br />
<br />
<br />
<b>Jedno wywołanie API — i z surowej tabeli Markdown dostajesz audytowalną strukturę z semantyką.</b> Strukturę, którą jutro podepniesz pod graf w Obsidianie.

Tak się buduje Poziom 3.

</p>
</details>

## Co osiągnęliśmy dzisiaj?

Spójrzmy, co masz teraz w rękach:

| Artefakt | Liczba | Co daje |
|---|---|---|
| **Dokumenty sklasyfikowane** | 7 | Nazwa spółki, okres, kategoria, mnożnik wartości, confidence |
| **Tabele sklasyfikowane** (próbka) | 10 | Tytuł, opis, tagi, pytania, wnioski, zastosowania |

Każdy z tych rekordów to **audytowalny artefakt** — wiesz, skąd pochodzi (document_id, table_id), masz confidence score, widzisz strukturę.

### To nie jest magiczny czat

Porównaj to z typowym "RAG-iem z tutoriala":

| Naiwny RAG | Nasz podejście |
|---|---|
| Wrzuć PDF → chunki → vector DB | Sparsuj PDF → wydobądź tabele → **sklasyfikuj każdą z metadanych** |
| Szukaj podobieństwa embeddingów | Szukaj po **tagach, pytaniach, tytule** (i ewentualnie embeddingach) |
| Model rediscoveruje wiedzę przy każdym zapytaniu | Wiedza jest **pre-skompilowana** i gotowa |
| Halucynacje trudno prześledzić | Każda odpowiedź ma **source** (document_id + table_id) |
| Kontekst = losowe chunki | Kontekst = **trafnie dobrane tabele** (jutro) |

<div class="alert alert-block alert-danger">

<h2>✅ Zaliczenie dnia 2</h2>
<h4>Co teraz zrobić?</h4>

<ul>
    <ol>📸 <b>Zrób Screenshot:</b> pokaż wynik jednego z zadań (2.1 lub 2.2) — ma być widać, że klasyfikator zadziałał i zwrócił strukturę.</ol>
    <ol>💬 <b>Wrzuć rozwiązanie na Slacka:</b> kanał <a href="https://dataworkshopclub.slack.com/archives/C0ADAKBRX3N"><code>#dwthon_llm_day2_done</code></a> — <b>tu wrzucamy tylko rozwiązania</b> (żeby łatwo było liczyć punkty i wydawać bonusy).</ol>
    <ol>❓ <b>Masz pytanie, utknąłeś, coś nie działa?</b> Pisz śmiało na kanał <a href="https://dataworkshopclub.slack.com/archives/C0AE19S8F96"><code>#dwthon_llm_day2</code></a> — <b>tu dyskutujemy, pomagamy, odpowiadamy</b>. Nie krępuj się, każde pytanie jest dobre.</ol>
    <ol>🆘 <b>Brak dostępu do Slacka?</b> Pisz na <code>anna@dataworkshop.eu</code>.</ol>
</ul>
</div>

### 🎁 Co dostajesz za oddane zadanie?

- **Dodatkowe materiały premium** na maila (rozszerzenie dzisiejszej lekcji)
- **50 punktów bonusowych** (zobaczysz w mailu, na co można je wymienić)
- **Krok bliżej certyfikatu** ukończenia DWthon

## Podsumowanie Dnia 2

Dzisiaj zrobiliśmy krok, którego większość rynku nie robi.

Nie wrzucaliśmy PDF-ów do bazy wektorowej licząc na cud. Zamiast tego budujemy **kontrolowany system**, który:

- wie, co to za dokument (spółka, okres, kategoria, mnożnik wartości),
- wie, co to za tabela (tytuł, opis, tagi, pytania),
- wie, na jakie pytania odpowiada,
- wie, ile jest pewien swojej odpowiedzi (confidence).

**To jest rzemiosło Software 3.0.** Nie prompt engineering. Nie "lepszy model". **Projektowanie struktury**, w której model pracuje.

### Co masz teraz w rękach

| Artefakt | Co daje |
|---|---|
| **`SimpleFinancialDocument` + `FinancialDocumentV2`** | Klasyfikator dokumentów — nazwa spółki, tytuł, okres, forma prawna |
| **`SimpleTableDetails` + `TableDetailsV2`** | Klasyfikator tabel — tytuł, opis, tagi, pytania biznesowe |
| **Porównanie Bielika i gpt-4o-mini** | Intuicja, kiedy który model wybrać |
| **Pełne wersje (z gwiazdką)** | Gotowy blueprint na produkcję |

### Co jutro (Dzień 3)?

Jutro ostatni dzień. Zbieramy to, co zbudowaliśmy dzisiaj, i robimy z tego **pełną Mapę Wiedzy**:

<img src="../images/graf_example.jpg" width="900px" />

*Tak wygląda cel — graf wiedzy w Obsidianie. Każdy węzeł to artykuł wygenerowany z metadanych. Linie to powiązania (spółka → dokument → tabela).*

Co konkretnie jutro:

1. **Gotowa klasyfikacja** — nie musisz uruchamiać klasyfikatorów na 1400+ tabelach. Zrobiłem to już za Ciebie. Dostaniesz gotowe dane.
2. **Generowanie artykułów Markdown** — pokażę, jak z metadanych zrobić strony, które Obsidian wyświetli jako graf
3. **Obsidian na żywo** — wrzucisz artykuły do swojego vaulta i zobaczysz graf w akcji
4. **Retrieval bez bazy wektorowej** — pokażę, jak zrobić retrieval używając Pandasa (i innych silników), a nie magicznej bazy wektorowej. To jest ta cała dyskusja "RAG umarł" — jutro zobaczysz, o co tak naprawdę chodzi.
5. **Chat z danymi** — zadajesz pytanie, system odpowiada z linkiem do źródła
6. **Mapa na GitHubie** — commit, push, i Mapa zostaje z Tobą. Edytowalna, rozszerzalna, Twoja.
7. Po ukończeniu 3 lekcji - otrzymujesz certyfikat :). 

Do zobaczenia.